Script to count number of classes

Imports

In [21]:
import pandas as pd

df = pd.read_csv('C:/repos/Blood-Cell-Classification/LabelledData/LabelExport_20260317.csv')

classes_name = ['Monocyte without RBC', 'Monocyte with RBC', 'Unusable', 'RBC alone', 'Clustered cell', 'Lymphocyte?']

for name in classes_name:
    count = (df['choice'] == name).sum()
    print(f"{name} count: {count}")

print(f"Number of labelled cells: {len(df)-1}")


Monocyte without RBC count: 5053
Monocyte with RBC count: 961
Unusable count: 2478
RBC alone count: 567
Clustered cell count: 7203
Lymphocyte? count: 175
Number of labelled cells: 16436


In [7]:
# Check if there are any nan labels
df[df['choice'].isna()]['id'].tolist()

[]

In [5]:
from pathlib import Path

root = Path(r"D:\MMA_LabelledData\Clustered_RBCCount")

for sub in root.iterdir():
    if sub.is_dir():
        count = sum(1 for _ in sub.iterdir())
        print(f"{sub.name}: {count} items")


RBC_0: 4980 items
RBC_1: 910 items
RBC_2: 458 items
RBC_3: 248 items
RBC_4: 41 items
RBC_5: 5 items
RBC_alone: 561 items


In [18]:
# Output the cumulative error
MonoNonMono = 0.9226
Clustered_class = 0.9381
Unclustered_RBCCount = 0.9375
Clustered_RBCCount = 0.9226
ThreeClass = 0.8634

RBCCount_Unclustered_Acc = Clustered_class * MonoNonMono * Unclustered_RBCCount
RBCCount_clustered_Acc = Clustered_class * MonoNonMono * Clustered_RBCCount 
ThreeClass_Clustered_Acc = ThreeClass * Clustered_RBCCount
ThreeClass_Unclustered_Acc = ThreeClass * Unclustered_RBCCount

print(f"Layered Classifiers Clustered: {RBCCount_clustered_Acc}")
print(f"Layered Classifiers Unclustered: {RBCCount_Unclustered_Acc}")
print(f"Three Class Clustered: {ThreeClass_Clustered_Acc}")
print(f"Three Class Unclustered: {ThreeClass_Unclustered_Acc}")

Layered Classifiers Clustered: 0.798502051956
Layered Classifiers Unclustered: 0.81139786875
Three Class Clustered: 0.79657284
Three Class Unclustered: 0.8094374999999999


In [42]:
import os
import json
from sklearn.model_selection import train_test_split

LABELLED_ROOT = r"D:\MMA_batch2_CellposeCustom\ContextCells"
OUTPUT_SPLITS = r"D:/MMA_LabelledData/splits"

os.makedirs(OUTPUT_SPLITS, exist_ok=True)

# Collect all labeled filenames
all_files = []
for class_folder in os.listdir(LABELLED_ROOT):
    class_path = os.path.join(LABELLED_ROOT, class_folder)
    if not os.path.isdir(class_path):
        continue
    for fname in os.listdir(class_path):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            all_files.append(fname)

print("Total labeled images:", len(all_files))


Total labeled images: 393337


In [29]:
# Build labels for stratification
labels = []
for fname in all_files:
    for class_folder in os.listdir(LABELLED_ROOT):
        if os.path.isfile(os.path.join(LABELLED_ROOT, class_folder, fname)):
            labels.append(class_folder)
            break

# First split: train (70%) vs temp (30%)
train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.30, stratify=labels, random_state=42
)

# Second split: val (10%) vs test (20%)
val_files, test_files = train_test_split(
    temp_files, test_size=2/3, stratify=temp_labels, random_state=42
)

print(len(train_files), len(val_files), len(test_files))

splits = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

with open(os.path.join(OUTPUT_SPLITS, "dataset_splits.json"), "w") as f:
    json.dump(splits, f, indent=2)

print("Saved dataset_splits.json")



11259 1608 3218
Saved dataset_splits.json


In [3]:
result = [{'final_pred': 1, 'final_score': 0.4995378255844116, 'path': [{'model': 'MonovsNonMono', 'pred': 1, 'score': 0.4995378255844116, 'probs': [0.2193862348794937, 0.4995378255844116, 0.2810758948326111]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0001.png'}, {'final_pred': 0, 'final_score': 0.8301953673362732, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.8301953673362732, 'probs': [0.8301953673362732, 0.001452345633879304, 0.16835227608680725]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0002.png'}, {'final_pred': 0, 'final_score': 0.986018180847168, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.986018180847168, 'probs': [0.986018180847168, 0.001515361713245511, 0.012466488406062126]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0003.png'}, {'final_pred': 0, 'final_score': 0.9864920973777771, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9864920973777771, 'probs': [0.9864920973777771, 0.0019416381837800145, 0.011566227301955223]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0004.png'}, {'final_pred': 3, 'final_score': 0.7064077258110046, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.668097972869873, 'probs': [0.3133552372455597, 0.018546834588050842, 0.668097972869873]}, {'model': 'Cluster', 'pred': 1, 'score': 0.520740807056427, 'probs': [0.479259192943573, 0.520740807056427]}, {'model': 'Cluster_RBCCount', 'pred': 3, 'score': 0.7064077258110046, 'probs': [0.0029736461583524942, 0.004127341788262129, 0.28442147374153137, 0.7064077258110046, 0.002069869777187705]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0005.png'}, {'final_pred': 2, 'final_score': 0.7570356130599976, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8441504836082458, 'probs': [0.15311099588871002, 0.002738469047471881, 0.8441504836082458]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9882264733314514, 'probs': [0.011773470789194107, 0.9882264733314514]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.7570356130599976, 'probs': [0.010106931440532207, 0.21841953694820404, 0.7570356130599976, 0.011414632201194763, 0.0030232961289584637]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0006.png'}, {'final_pred': 0, 'final_score': 0.9959409236907959, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9488096237182617, 'probs': [0.04646905139088631, 0.004721262026578188, 0.9488096237182617]}, {'model': 'Cluster', 'pred': 0, 'score': 0.984672486782074, 'probs': [0.984672486782074, 0.015327556990087032]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9959409236907959, 'probs': [0.9959409236907959, 0.0033812953624874353, 0.0002728218387346715, 0.00040494208224117756]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0007.png'}, {'final_pred': 0, 'final_score': 0.4752960205078125, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.513067364692688, 'probs': [0.48223862051963806, 0.004694011993706226, 0.513067364692688]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9871410131454468, 'probs': [0.9871410131454468, 0.012859013862907887]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.4752960205078125, 'probs': [0.4752960205078125, 0.36807775497436523, 0.06334581226110458, 0.0932803675532341]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0008.png'}, {'final_pred': 0, 'final_score': 0.9957224130630493, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9503234028816223, 'probs': [0.04097197949886322, 0.008704716339707375, 0.9503234028816223]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9899103045463562, 'probs': [0.9899103045463562, 0.010089711286127567]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9957224130630493, 'probs': [0.9957224130630493, 0.003565033432096243, 0.0002576615661382675, 0.00045488475007005036]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0009.png'}, {'final_pred': 1, 'final_score': 0.9988899827003479, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.996238112449646, 'probs': [0.002375851385295391, 0.0013860617764294147, 0.996238112449646]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9984906911849976, 'probs': [0.9984906911849976, 0.0015092488611117005]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9988899827003479, 'probs': [0.0004596240760292858, 0.9988899827003479, 0.0005889548338018358, 6.149921682663262e-05]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0010.png'}, {'final_pred': 0, 'final_score': 0.8548577427864075, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9489331841468811, 'probs': [0.03425160050392151, 0.01681520603597164, 0.9489331841468811]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9882515072822571, 'probs': [0.9882515072822571, 0.011748431250452995]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.8548577427864075, 'probs': [0.8548577427864075, 0.13960620760917664, 0.003040885552763939, 0.002495145658031106]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0011.png'}, {'final_pred': 0, 'final_score': 0.9976958632469177, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.940189778804779, 'probs': [0.044258296489715576, 0.015551948919892311, 0.940189778804779]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9983620047569275, 'probs': [0.9983620047569275, 0.0016380317974835634]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9976958632469177, 'probs': [0.9976958632469177, 0.001626298762857914, 0.00017426078557036817, 0.0005035670474171638]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0012.png'}, {'final_pred': 1, 'final_score': 0.9690365195274353, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9800758957862854, 'probs': [0.01389478612691164, 0.006029386073350906, 0.9800758957862854]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9991753697395325, 'probs': [0.9991753697395325, 0.0008246577926911414]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9690365195274353, 'probs': [0.001335359294898808, 0.9690365195274353, 0.029308518394827843, 0.00031964649679139256]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0013.png'}, {'final_pred': 0, 'final_score': 0.9952942728996277, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.5949860215187073, 'probs': [0.40116989612579346, 0.003844057908281684, 0.5949860215187073]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9248207807540894, 'probs': [0.9248207807540894, 0.07517924904823303]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9952942728996277, 'probs': [0.9952942728996277, 0.0032368297688663006, 0.0005519941914826632, 0.0009169425466097891]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0014.png'}, {'final_pred': 0, 'final_score': 0.953405499458313, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9636631608009338, 'probs': [0.03551227226853371, 0.0008246071520261467, 0.9636631608009338]}, {'model': 'Cluster', 'pred': 1, 'score': 0.7505596876144409, 'probs': [0.2494402676820755, 0.7505596876144409]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.953405499458313, 'probs': [0.953405499458313, 0.040245410054922104, 0.0037220530211925507, 0.0014643832109868526, 0.0011625937186181545]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0015.png'}, {'final_pred': 1, 'final_score': 0.41145122051239014, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9581345915794373, 'probs': [0.025566911324858665, 0.016298580914735794, 0.9581345915794373]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9038567543029785, 'probs': [0.9038567543029785, 0.09614323079586029]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.41145122051239014, 'probs': [0.382718563079834, 0.41145122051239014, 0.1402776837348938, 0.06555254012346268]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0016.png'}, {'final_pred': 0, 'final_score': 0.738704264163971, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.738704264163971, 'probs': [0.738704264163971, 0.0013676309026777744, 0.2599281072616577]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0017.png'}, {'final_pred': 0, 'final_score': 0.9970863461494446, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9662582874298096, 'probs': [0.01766831986606121, 0.016073426231741905, 0.9662582874298096]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9984795451164246, 'probs': [0.9984795451164246, 0.0015204541850835085]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9970863461494446, 'probs': [0.9970863461494446, 0.002075288910418749, 0.0002850276359822601, 0.000553299265448004]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0018.png'}, {'final_pred': 2, 'final_score': 0.5180270671844482, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9532148241996765, 'probs': [0.040351685136556625, 0.006433493923395872, 0.9532148241996765]}, {'model': 'Cluster', 'pred': 1, 'score': 0.992973804473877, 'probs': [0.0070262327790260315, 0.992973804473877]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.5180270671844482, 'probs': [0.032086752355098724, 0.32707056403160095, 0.5180270671844482, 0.09726870059967041, 0.025546900928020477]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0019.png'}, {'final_pred': 0, 'final_score': 0.9616486430168152, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9228553175926208, 'probs': [0.05128476396203041, 0.025860019028186798, 0.9228553175926208]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9886386394500732, 'probs': [0.9886386394500732, 0.011361334472894669]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9616486430168152, 'probs': [0.9616486430168152, 0.026130342856049538, 0.0038461191579699516, 0.008374970406293869]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0020.png'}, {'final_pred': 1, 'final_score': 0.530536949634552, 'path': [{'model': 'MonovsNonMono', 'pred': 1, 'score': 0.530536949634552, 'probs': [0.3284919857978821, 0.530536949634552, 0.14097106456756592]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0021.png'}, {'final_pred': 0, 'final_score': 0.9959410429000854, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9575151205062866, 'probs': [0.03770811855792999, 0.004776677582412958, 0.9575151205062866]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9836595058441162, 'probs': [0.9836595058441162, 0.01634052023291588]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9959410429000854, 'probs': [0.9959410429000854, 0.0031009165104478598, 0.00030041069840081036, 0.0006577508756890893]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0022.png'}, {'final_pred': 1, 'final_score': 0.8903093934059143, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9831280708312988, 'probs': [0.00887990277260542, 0.007992134429514408, 0.9831280708312988]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9984844326972961, 'probs': [0.9984844326972961, 0.0015155973378568888]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.8903093934059143, 'probs': [0.003406318137422204, 0.8903093934059143, 0.10505856573581696, 0.0012257384369149804]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0023.png'}, {'final_pred': 2, 'final_score': 0.6850730180740356, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7858025431632996, 'probs': [0.21050991117954254, 0.0036875582300126553, 0.7858025431632996]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9834387898445129, 'probs': [0.0165612380951643, 0.9834387898445129]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.6850730180740356, 'probs': [0.010372664779424667, 0.013369431719183922, 0.6850730180740356, 0.27029016613960266, 0.020894730463624]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0024.png'}, {'final_pred': 2, 'final_score': 0.5484545826911926, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.6457105278968811, 'probs': [0.3428921401500702, 0.011397363618016243, 0.6457105278968811]}, {'model': 'Cluster', 'pred': 1, 'score': 0.8959311842918396, 'probs': [0.1040688082575798, 0.8959311842918396]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.5484545826911926, 'probs': [0.013201667927205563, 0.054759275168180466, 0.5484545826911926, 0.3809363842010498, 0.002648093970492482]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0025.png'}, {'final_pred': 2, 'final_score': 0.846272349357605, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7600522637367249, 'probs': [0.23851555585861206, 0.0014322203351184726, 0.7600522637367249]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9217542409896851, 'probs': [0.07824572920799255, 0.9217542409896851]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.846272349357605, 'probs': [0.007082619704306126, 0.10416800528764725, 0.846272349357605, 0.030972443521022797, 0.011504589579999447]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0026.png'}, {'final_pred': 2, 'final_score': 0.7097311019897461, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9630643129348755, 'probs': [0.022730743512511253, 0.014204928651452065, 0.9630643129348755]}, {'model': 'Cluster', 'pred': 1, 'score': 0.7270417213439941, 'probs': [0.27295827865600586, 0.7270417213439941]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.7097311019897461, 'probs': [0.008261322975158691, 0.012996179983019829, 0.7097311019897461, 0.26711902022361755, 0.0018924266332760453]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0027.png'}, {'final_pred': 0, 'final_score': 0.9711766242980957, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9548924565315247, 'probs': [0.03682779520750046, 0.008279784582555294, 0.9548924565315247]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9557580351829529, 'probs': [0.9557580351829529, 0.04424196109175682]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9711766242980957, 'probs': [0.9711766242980957, 0.02710827626287937, 0.0009685917757451534, 0.0007465826929546893]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0028.png'}, {'final_pred': 0, 'final_score': 0.9866243600845337, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9227893948554993, 'probs': [0.055856093764305115, 0.021354472264647484, 0.9227893948554993]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9718822240829468, 'probs': [0.9718822240829468, 0.02811778523027897]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9866243600845337, 'probs': [0.9866243600845337, 0.009399915114045143, 0.0011072374181821942, 0.0028684830758720636]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0029.png'}, {'final_pred': 1, 'final_score': 0.9779185652732849, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.964089035987854, 'probs': [0.02034183032810688, 0.015569129027426243, 0.964089035987854]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9964172840118408, 'probs': [0.9964172840118408, 0.0035827532410621643]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9779185652732849, 'probs': [0.008720926940441132, 0.9779185652732849, 0.01270872913300991, 0.000651798618491739]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0030.png'}, {'final_pred': 0, 'final_score': 0.9968326687812805, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9331250786781311, 'probs': [0.05941885709762573, 0.007456021383404732, 0.9331250786781311]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9872345328330994, 'probs': [0.9872345328330994, 0.012765473686158657]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9968326687812805, 'probs': [0.9968326687812805, 0.0018344438867643476, 0.00024600676260888577, 0.0010868596145883203]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0031.png'}, {'final_pred': 1, 'final_score': 0.8920449614524841, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9651334285736084, 'probs': [0.026313507929444313, 0.008553083054721355, 0.9651334285736084]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9371776580810547, 'probs': [0.9371776580810547, 0.06282226741313934]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.8920449614524841, 'probs': [0.07900125533342361, 0.8920449614524841, 0.02273024059832096, 0.006223558913916349]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0032.png'}, {'final_pred': 0, 'final_score': 0.6573436260223389, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9146416187286377, 'probs': [0.07127006351947784, 0.014088393189013004, 0.9146416187286377]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9750930070877075, 'probs': [0.9750930070877075, 0.024906961247324944]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.6573436260223389, 'probs': [0.6573436260223389, 0.31081628799438477, 0.019317952916026115, 0.012522086501121521]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0033.png'}, {'final_pred': 0, 'final_score': 0.9980712532997131, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9466763734817505, 'probs': [0.04133986309170723, 0.01198386400938034, 0.9466763734817505]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9791767001152039, 'probs': [0.9791767001152039, 0.020823325961828232]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9980712532997131, 'probs': [0.9980712532997131, 0.001178249716758728, 0.00017054274212568998, 0.0005800591316074133]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0034.png'}, {'final_pred': 0, 'final_score': 0.9956966638565063, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9095365405082703, 'probs': [0.08068683743476868, 0.009776754304766655, 0.9095365405082703]}, {'model': 'Cluster', 'pred': 0, 'score': 0.7681723237037659, 'probs': [0.7681723237037659, 0.23182767629623413]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9956966638565063, 'probs': [0.9956966638565063, 0.003737357212230563, 0.00025206536520272493, 0.00031387730268761516]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0035.png'}, {'final_pred': 0, 'final_score': 0.9977994561195374, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9170823097229004, 'probs': [0.07509522885084152, 0.007822444662451744, 0.9170823097229004]}, {'model': 'Cluster', 'pred': 0, 'score': 0.6749908328056335, 'probs': [0.6749908328056335, 0.32500913739204407]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9977994561195374, 'probs': [0.9977994561195374, 0.0013714778469875455, 0.00019363811588846147, 0.0006354128709062934]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0036.png'}, {'final_pred': 0, 'final_score': 0.9930508136749268, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7887417674064636, 'probs': [0.20862959325313568, 0.0026285985950380564, 0.7887417674064636]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9622676968574524, 'probs': [0.9622676968574524, 0.03773228079080582]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9930508136749268, 'probs': [0.9930508136749268, 0.005182594060897827, 0.0007327162893489003, 0.001033902633935213]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0037.png'}, {'final_pred': 0, 'final_score': 0.9973145127296448, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9485777616500854, 'probs': [0.04347221180796623, 0.007950061932206154, 0.9485777616500854]}, {'model': 'Cluster', 'pred': 0, 'score': 0.709701418876648, 'probs': [0.709701418876648, 0.2902986407279968]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9973145127296448, 'probs': [0.9973145127296448, 0.0022518294863402843, 0.0001495328324381262, 0.0002840853703673929]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0038.png'}, {'final_pred': 0, 'final_score': 0.7121521234512329, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9480792284011841, 'probs': [0.0287270899862051, 0.023193731904029846, 0.9480792284011841]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9907222986221313, 'probs': [0.9907222986221313, 0.0092777656391263]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.7121521234512329, 'probs': [0.7121521234512329, 0.2632024586200714, 0.010859871283173561, 0.013785595074295998]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0039.png'}, {'final_pred': 0, 'final_score': 0.9967483282089233, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9527729153633118, 'probs': [0.0365675687789917, 0.010659554973244667, 0.9527729153633118]}, {'model': 'Cluster', 'pred': 0, 'score': 0.8980227112770081, 'probs': [0.8980227112770081, 0.10197723656892776]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9967483282089233, 'probs': [0.9967483282089233, 0.0026283643674105406, 0.0002090468624373898, 0.00041435318416915834]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0040.png'}, {'final_pred': 1, 'final_score': 0.9666594862937927, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8576123118400574, 'probs': [0.10007988661527634, 0.04230772331357002, 0.8576123118400574]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9928326606750488, 'probs': [0.9928326606750488, 0.007167381700128317]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9666594862937927, 'probs': [0.005931403022259474, 0.9666594862937927, 0.02673223987221718, 0.0006769315805286169]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0041.png'}, {'final_pred': 1, 'final_score': 0.6316416263580322, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9549750685691833, 'probs': [0.040951207280159, 0.00407367991283536, 0.9549750685691833]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9769009947776794, 'probs': [0.023098977282643318, 0.9769009947776794]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.6316416263580322, 'probs': [0.02117159217596054, 0.6316416263580322, 0.3375844359397888, 0.0035172279458492994, 0.006085122935473919]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0042.png'}, {'final_pred': 0, 'final_score': 0.9463185667991638, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9853443503379822, 'probs': [0.013456633314490318, 0.001199070131406188, 0.9853443503379822]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9973076581954956, 'probs': [0.002692344132810831, 0.9973076581954956]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9463185667991638, 'probs': [0.9463185667991638, 0.04858437180519104, 0.0027126988861709833, 0.0007899630581960082, 0.0015943697653710842]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0043.png'}, {'final_pred': 1, 'final_score': 0.7328498959541321, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9695785045623779, 'probs': [0.012581296265125275, 0.017840152606368065, 0.9695785045623779]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9971219897270203, 'probs': [0.9971219897270203, 0.0028779618442058563]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.7328498959541321, 'probs': [0.2445683479309082, 0.7328498959541321, 0.01744818687438965, 0.005133620463311672]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0044.png'}, {'final_pred': 0, 'final_score': 0.8674893975257874, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.5748047232627869, 'probs': [0.4203301668167114, 0.004865162540227175, 0.5748047232627869]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9814101457595825, 'probs': [0.01858988218009472, 0.9814101457595825]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.8674893975257874, 'probs': [0.8674893975257874, 0.1205902099609375, 0.007575846277177334, 0.002690558321774006, 0.0016539698699489236]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0045.png'}, {'final_pred': 1, 'final_score': 0.8597478866577148, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8994297385215759, 'probs': [0.09899229556322098, 0.001577913761138916, 0.8994297385215759]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9991475343704224, 'probs': [0.0008524903678335249, 0.9991475343704224]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.8597478866577148, 'probs': [0.08039049804210663, 0.8597478866577148, 0.0570555105805397, 0.0017913634655997157, 0.0010148367146030068]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0046.png'}, {'final_pred': 1, 'final_score': 0.9864479303359985, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9679601788520813, 'probs': [0.03018368035554886, 0.0018561274046078324, 0.9679601788520813]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9959264993667603, 'probs': [0.004073494579643011, 0.9959264993667603]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.9864479303359985, 'probs': [0.010647201910614967, 0.9864479303359985, 0.001753558753989637, 9.652072185417637e-05, 0.0010547628626227379]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0047.png'}, {'final_pred': 1, 'final_score': 0.9654215574264526, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9570444226264954, 'probs': [0.037842731922864914, 0.005112850107252598, 0.9570444226264954]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9982994198799133, 'probs': [0.0017005912959575653, 0.9982994198799133]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.9654215574264526, 'probs': [0.013367220759391785, 0.9654215574264526, 0.019914090633392334, 0.00025456046569161117, 0.0010425786022096872]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0048.png'}, {'final_pred': 1, 'final_score': 0.7016063928604126, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9529470205307007, 'probs': [0.0428224615752697, 0.004230435471981764, 0.9529470205307007]}, {'model': 'Cluster', 'pred': 1, 'score': 0.906546950340271, 'probs': [0.09345301985740662, 0.906546950340271]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.7016063928604126, 'probs': [0.015236523933708668, 0.7016063928604126, 0.24968890845775604, 0.02474283054471016, 0.008725373074412346]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0049.png'}, {'final_pred': 0, 'final_score': 0.9267149567604065, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9307203888893127, 'probs': [0.060052722692489624, 0.009226816706359386, 0.9307203888893127]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9914472103118896, 'probs': [0.9914472103118896, 0.008552744053304195]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9267149567604065, 'probs': [0.9267149567604065, 0.06880137324333191, 0.003121388843283057, 0.0013622684637084603]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0050.png'}, {'final_pred': 0, 'final_score': 0.9067042469978333, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8935010433197021, 'probs': [0.09369835257530212, 0.012800617143511772, 0.8935010433197021]}, {'model': 'Cluster', 'pred': 0, 'score': 0.99225252866745, 'probs': [0.99225252866745, 0.007747478783130646]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9067042469978333, 'probs': [0.9067042469978333, 0.08331950753927231, 0.0039182729087769985, 0.006057938560843468]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0051.png'}, {'final_pred': 1, 'final_score': 0.9970024228096008, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9911558628082275, 'probs': [0.006516215391457081, 0.0023279027082026005, 0.9911558628082275]}, {'model': 'Cluster', 'pred': 0, 'score': 0.7471916079521179, 'probs': [0.7471916079521179, 0.25280842185020447]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9970024228096008, 'probs': [0.0005350891151465476, 0.9970024228096008, 0.002347883302718401, 0.0001145619316957891]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0052.png'}, {'final_pred': 0, 'final_score': 0.9444257020950317, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8641299605369568, 'probs': [0.054999642074108124, 0.08087046444416046, 0.8641299605369568]}, {'model': 'Cluster', 'pred': 0, 'score': 0.5009991526603699, 'probs': [0.5009991526603699, 0.49900078773498535]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9444257020950317, 'probs': [0.9444257020950317, 0.02470674365758896, 0.0072533367201685905, 0.023614222183823586]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0053.png'}, {'final_pred': 1, 'final_score': 0.9569855332374573, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.963897168636322, 'probs': [0.030670620501041412, 0.005432303994894028, 0.963897168636322]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9852919578552246, 'probs': [0.9852919578552246, 0.01470805611461401]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9569855332374573, 'probs': [0.017026182264089584, 0.9569855332374573, 0.023594947531819344, 0.0023933674674481153]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0054.png'}, {'final_pred': 0, 'final_score': 0.9977492690086365, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9450560808181763, 'probs': [0.04141341894865036, 0.013530558906495571, 0.9450560808181763]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9963371753692627, 'probs': [0.9963371753692627, 0.0036627966910600662]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9977492690086365, 'probs': [0.9977492690086365, 0.001597581198439002, 0.0001888792758109048, 0.00046430787188000977]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0055.png'}, {'final_pred': 2, 'final_score': 0.9900667667388916, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9895642995834351, 'probs': [0.00508847227320075, 0.005347286816686392, 0.9895642995834351]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9970876574516296, 'probs': [0.9970876574516296, 0.0029123472049832344]}, {'model': 'Unclustered_RBCCount', 'pred': 2, 'score': 0.9900667667388916, 'probs': [0.00030541224987246096, 0.0021575901191681623, 0.9900667667388916, 0.007470158860087395]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0056.png'}, {'final_pred': 1, 'final_score': 0.8720212578773499, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7681930065155029, 'probs': [0.22874510288238525, 0.0030618126038461924, 0.7681930065155029]}, {'model': 'Cluster', 'pred': 1, 'score': 0.7704768180847168, 'probs': [0.22952322661876678, 0.7704768180847168]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.8720212578773499, 'probs': [0.06483929604291916, 0.8720212578773499, 0.06116326153278351, 0.0007825290667824447, 0.0011936582159250975]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0057.png'}, {'final_pred': 0, 'final_score': 0.7193936109542847, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9501189589500427, 'probs': [0.046926286071538925, 0.002954653697088361, 0.9501189589500427]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9952723383903503, 'probs': [0.004727637395262718, 0.9952723383903503]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.7193936109542847, 'probs': [0.7193936109542847, 0.2498473972082138, 0.026640713214874268, 0.0014415299519896507, 0.0026767575182020664]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0058.png'}, {'final_pred': 1, 'final_score': 0.9970055222511292, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9845643043518066, 'probs': [0.011608462780714035, 0.0038271313533186913, 0.9845643043518066]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9974631071090698, 'probs': [0.9974631071090698, 0.0025368868373334408]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9970055222511292, 'probs': [0.0005646086065098643, 0.9970055222511292, 0.002349923364818096, 7.998612272785977e-05]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0059.png'}, {'final_pred': 4, 'final_score': 0.7943109273910522, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.6481174826622009, 'probs': [0.3465125560760498, 0.005369973368942738, 0.6481174826622009]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9995137453079224, 'probs': [0.0004862169735133648, 0.9995137453079224]}, {'model': 'Cluster_RBCCount', 'pred': 4, 'score': 0.7943109273910522, 'probs': [0.0014873004984110594, 0.19899892807006836, 0.0046365028247237206, 0.000566394068300724, 0.7943109273910522]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0060.png'}, {'final_pred': 0, 'final_score': 0.992031455039978, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9491561651229858, 'probs': [0.04169701784849167, 0.009146741591393948, 0.9491561651229858]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9987275004386902, 'probs': [0.9987275004386902, 0.0012724809348583221]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.992031455039978, 'probs': [0.992031455039978, 0.006833996158093214, 0.0005457994411699474, 0.0005888037267141044]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0061.png'}, {'final_pred': 1, 'final_score': 0.9986404776573181, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9947441816329956, 'probs': [0.0027257574256509542, 0.0025300488341599703, 0.9947441816329956]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9983789920806885, 'probs': [0.9983789920806885, 0.0016210026806220412]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9986404776573181, 'probs': [0.0004329912771936506, 0.9986404776573181, 0.0008566734613850713, 6.992114504100755e-05]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0062.png'}, {'final_pred': 2, 'final_score': 0.8557196855545044, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9381116032600403, 'probs': [0.048912711441516876, 0.012975702993571758, 0.9381116032600403]}, {'model': 'Cluster', 'pred': 1, 'score': 0.7157658934593201, 'probs': [0.28423407673835754, 0.7157658934593201]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.8557196855545044, 'probs': [0.007783402688801289, 0.1250530332326889, 0.8557196855545044, 0.009040312841534615, 0.0024035645183175802]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0063.png'}, {'final_pred': 0, 'final_score': 0.9570698142051697, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9771717190742493, 'probs': [0.020314669236540794, 0.0025135294999927282, 0.9771717190742493]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9852442741394043, 'probs': [0.014755737036466599, 0.9852442741394043]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9570698142051697, 'probs': [0.9570698142051697, 0.028770511969923973, 0.009554879739880562, 0.0033100121654570103, 0.0012948332587257028]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0064.png'}, {'final_pred': 1, 'final_score': 0.8765753507614136, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9685630202293396, 'probs': [0.02348574437201023, 0.007951282896101475, 0.9685630202293396]}, {'model': 'Cluster', 'pred': 0, 'score': 0.8825290203094482, 'probs': [0.8825290203094482, 0.11747095733880997]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.8765753507614136, 'probs': [0.08614102751016617, 0.8765753507614136, 0.03385380282998085, 0.0034297192469239235]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0065.png'}, {'final_pred': 0, 'final_score': 0.9704828262329102, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9704828262329102, 'probs': [0.9704828262329102, 0.006204142235219479, 0.0233130045235157]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0066.png'}, {'final_pred': 1, 'final_score': 0.950438380241394, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9664965271949768, 'probs': [0.02592496946454048, 0.007578474935144186, 0.9664965271949768]}, {'model': 'Cluster', 'pred': 0, 'score': 0.6147734522819519, 'probs': [0.6147734522819519, 0.3852265477180481]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.950438380241394, 'probs': [0.038785841315984726, 0.950438380241394, 0.008476356975734234, 0.0022994137834757566]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0067.png'}, {'final_pred': 1, 'final_score': 0.629207968711853, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9863526821136475, 'probs': [0.012848027981817722, 0.0007993644685484469, 0.9863526821136475]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9995648264884949, 'probs': [0.00043516739970073104, 0.9995648264884949]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.629207968711853, 'probs': [0.20465023815631866, 0.629207968711853, 0.13247759640216827, 0.008809879422187805, 0.024854324758052826]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0068.png'}, {'final_pred': 1, 'final_score': 0.7699105143547058, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.881424605846405, 'probs': [0.11501917988061905, 0.0035562575794756413, 0.881424605846405]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9729156494140625, 'probs': [0.02708440274000168, 0.9729156494140625]}, {'model': 'Cluster_RBCCount', 'pred': 1, 'score': 0.7699105143547058, 'probs': [0.08639011532068253, 0.7699105143547058, 0.10934356600046158, 0.014610528945922852, 0.01974528469145298]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0069.png'}, {'final_pred': 1, 'final_score': 0.998773992061615, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9947798252105713, 'probs': [0.0029383457731455564, 0.0022818678990006447, 0.9947798252105713]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9973862767219543, 'probs': [0.9973862767219543, 0.0026137277018278837]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.998773992061615, 'probs': [0.0004629042523447424, 0.998773992061615, 0.0006930891540832818, 6.999949255259708e-05]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0070.png'}, {'final_pred': 2, 'final_score': 0.8288525938987732, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7925470471382141, 'probs': [0.20402197539806366, 0.0034310168121010065, 0.7925470471382141]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9984177350997925, 'probs': [0.0015823127469047904, 0.9984177350997925]}, {'model': 'Cluster_RBCCount', 'pred': 2, 'score': 0.8288525938987732, 'probs': [0.0071141705848276615, 0.13794304430484772, 0.8288525938987732, 0.010423470288515091, 0.015666721388697624]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0071.png'}, {'final_pred': 0, 'final_score': 0.9964942336082458, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9621239900588989, 'probs': [0.03125733509659767, 0.006618687883019447, 0.9621239900588989]}, {'model': 'Cluster', 'pred': 0, 'score': 0.993496298789978, 'probs': [0.993496298789978, 0.0065036253072321415]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9964942336082458, 'probs': [0.9964942336082458, 0.0029391911812126637, 0.00025420659221708775, 0.0003123331116512418]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0072.png'}, {'final_pred': 0, 'final_score': 0.9752823114395142, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.6119117736816406, 'probs': [0.2224927395582199, 0.16559551656246185, 0.6119117736816406]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9914780259132385, 'probs': [0.9914780259132385, 0.008521948009729385]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9752823114395142, 'probs': [0.9752823114395142, 0.022813381627202034, 0.001102227601222694, 0.0008021829416975379]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0073.png'}, {'final_pred': 0, 'final_score': 0.9934971332550049, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8160104155540466, 'probs': [0.057531122118234634, 0.1264585256576538, 0.8160104155540466]}, {'model': 'Cluster', 'pred': 0, 'score': 0.993740975856781, 'probs': [0.993740975856781, 0.006259014364331961]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9934971332550049, 'probs': [0.9934971332550049, 0.0050978693179786205, 0.0005158448475413024, 0.0008890582830645144]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0074.png'}, {'final_pred': 0, 'final_score': 0.9930094480514526, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9458833336830139, 'probs': [0.04548383876681328, 0.008632881566882133, 0.9458833336830139]}, {'model': 'Cluster', 'pred': 0, 'score': 0.994498074054718, 'probs': [0.994498074054718, 0.0055019427090883255]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9930094480514526, 'probs': [0.9930094480514526, 0.00591423362493515, 0.0003486061468720436, 0.0007277496624737978]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0075.png'}, {'final_pred': 0, 'final_score': 0.971977174282074, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8428593277931213, 'probs': [0.15419481694698334, 0.0029458305798470974, 0.8428593277931213]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9880440831184387, 'probs': [0.9880440831184387, 0.01195585262030363]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.971977174282074, 'probs': [0.971977174282074, 0.0234299935400486, 0.0013559143990278244, 0.0032369401305913925]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0076.png'}, {'final_pred': 0, 'final_score': 0.9689619541168213, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8580474853515625, 'probs': [0.054657842963933945, 0.08729470521211624, 0.8580474853515625]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9878233671188354, 'probs': [0.9878233671188354, 0.012176628224551678]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9689619541168213, 'probs': [0.9689619541168213, 0.014950549229979515, 0.003336153691634536, 0.012751287780702114]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0077.png'}, {'final_pred': 0, 'final_score': 0.9981986880302429, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9642388224601746, 'probs': [0.026081716641783714, 0.00967938918620348, 0.9642388224601746]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9961416125297546, 'probs': [0.9961416125297546, 0.00385839375667274]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9981986880302429, 'probs': [0.9981986880302429, 0.0011913724010810256, 0.00021180184558033943, 0.00039815387572161853]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0078.png'}, {'final_pred': 0, 'final_score': 0.9918301701545715, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9918301701545715, 'probs': [0.9918301701545715, 0.0005321754142642021, 0.007637618109583855]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0079.png'}, {'final_pred': 0, 'final_score': 0.9234579801559448, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9234579801559448, 'probs': [0.9234579801559448, 0.001002581906504929, 0.07553945481777191]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0080.png'}, {'final_pred': 0, 'final_score': 0.879322350025177, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.469239205121994, 'probs': [0.11871936172246933, 0.41204139590263367, 0.469239205121994]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9955719709396362, 'probs': [0.9955719709396362, 0.004427976440638304]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.879322350025177, 'probs': [0.879322350025177, 0.1169140562415123, 0.0029043161775916815, 0.0008592456579208374]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0081.png'}, {'final_pred': 0, 'final_score': 0.9616217017173767, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8909025192260742, 'probs': [0.10763894766569138, 0.0014585363678634167, 0.8909025192260742]}, {'model': 'Cluster', 'pred': 1, 'score': 0.9682934284210205, 'probs': [0.03170657530426979, 0.9682934284210205]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9616217017173767, 'probs': [0.9616217017173767, 0.028970614075660706, 0.004855679348111153, 0.0017294916324317455, 0.0028225018177181482]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0001.png'}, {'final_pred': 0, 'final_score': 0.9896466135978699, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9573281407356262, 'probs': [0.038751937448978424, 0.003919913899153471, 0.9573281407356262]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9331678748130798, 'probs': [0.9331678748130798, 0.06683212518692017]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9896466135978699, 'probs': [0.9896466135978699, 0.008823500014841557, 0.0006447960040532053, 0.0008849945734255016]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0002.png'}, {'final_pred': 0, 'final_score': 0.9960493445396423, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9438860416412354, 'probs': [0.051801543682813644, 0.004312453791499138, 0.9438860416412354]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9933149218559265, 'probs': [0.9933149218559265, 0.00668507581576705]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9960493445396423, 'probs': [0.9960493445396423, 0.0032252613455057144, 0.00024922078591771424, 0.0004762113094329834]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0003.png'}, {'final_pred': 0, 'final_score': 0.5821899771690369, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.5821899771690369, 'probs': [0.5821899771690369, 0.0013197349617257714, 0.41649025678634644]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0004.png'}, {'final_pred': 1, 'final_score': 0.9097294807434082, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8321349024772644, 'probs': [0.1064731702208519, 0.061391931027173996, 0.8321349024772644]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9842544794082642, 'probs': [0.9842544794082642, 0.01574554108083248]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9097294807434082, 'probs': [0.08047979325056076, 0.9097294807434082, 0.007863515056669712, 0.0019272833596915007]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0005.png'}, {'final_pred': 0, 'final_score': 0.9937129616737366, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9014431238174438, 'probs': [0.09495284408330917, 0.003604054916650057, 0.9014431238174438]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9747615456581116, 'probs': [0.9747615456581116, 0.025238480418920517]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9937129616737366, 'probs': [0.9937129616737366, 0.005291963927447796, 0.00038059643702581525, 0.0006144815124571323]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0006.png'}, {'final_pred': 0, 'final_score': 0.9979068040847778, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9169817566871643, 'probs': [0.0802604928612709, 0.0027577602304518223, 0.9169817566871643]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9964518547058105, 'probs': [0.9964518547058105, 0.003548107575625181]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9979068040847778, 'probs': [0.9979068040847778, 0.0015901840524747968, 0.0002073852374451235, 0.0002957111573778093]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0007.png'}, {'final_pred': 0, 'final_score': 0.9963765740394592, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9532091021537781, 'probs': [0.04206441715359688, 0.004726459737867117, 0.9532091021537781]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9297223091125488, 'probs': [0.9297223091125488, 0.07027769833803177]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9963765740394592, 'probs': [0.9963765740394592, 0.0029090421739965677, 0.00024973193649202585, 0.0004645644803531468]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0008.png'}, {'final_pred': 0, 'final_score': 0.9971355199813843, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8482100963592529, 'probs': [0.15016897022724152, 0.0016208253800868988, 0.8482100963592529]}, {'model': 'Cluster', 'pred': 0, 'score': 0.912529468536377, 'probs': [0.912529468536377, 0.08747055381536484]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9971355199813843, 'probs': [0.9971355199813843, 0.002329218899831176, 0.00018332555191591382, 0.00035200268030166626]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0009.png'}, {'final_pred': 0, 'final_score': 0.969993531703949, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.969993531703949, 'probs': [0.969993531703949, 0.00260759424418211, 0.02739883027970791]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0010.png'}, {'final_pred': 0, 'final_score': 0.9909731149673462, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8873249888420105, 'probs': [0.11030907183885574, 0.0023658904246985912, 0.8873249888420105]}, {'model': 'Cluster', 'pred': 0, 'score': 0.992651641368866, 'probs': [0.992651641368866, 0.00734832976013422]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9909731149673462, 'probs': [0.9909731149673462, 0.006672343239188194, 0.0007823535706847906, 0.0015721170930191875]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0011.png'}, {'final_pred': 0, 'final_score': 0.7864611744880676, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8744112253189087, 'probs': [0.12002161145210266, 0.0055671376176178455, 0.8744112253189087]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9970449805259705, 'probs': [0.9970449805259705, 0.002955057891085744]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.7864611744880676, 'probs': [0.7864611744880676, 0.20838843286037445, 0.0036298076156526804, 0.0015205093659460545]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0012.png'}, {'final_pred': 0, 'final_score': 0.9553735852241516, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9553735852241516, 'probs': [0.9553735852241516, 0.0024712206795811653, 0.04215523973107338]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0013.png'}, {'final_pred': 0, 'final_score': 0.9900692105293274, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8930433392524719, 'probs': [0.10555271804332733, 0.0014040220994502306, 0.8930433392524719]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9891555905342102, 'probs': [0.9891555905342102, 0.010844483971595764]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9900692105293274, 'probs': [0.9900692105293274, 0.008773023262619972, 0.00029528644517995417, 0.0008624775218777359]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0014.png'}, {'final_pred': 0, 'final_score': 0.9882978796958923, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9142256379127502, 'probs': [0.07750418037176132, 0.008270165883004665, 0.9142256379127502]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9881821274757385, 'probs': [0.9881821274757385, 0.011817875318229198]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9882978796958923, 'probs': [0.9882978796958923, 0.009391147643327713, 0.0006331215845420957, 0.0016778892604634166]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0015.png'}, {'final_pred': 1, 'final_score': 0.877342700958252, 'path': [{'model': 'MonovsNonMono', 'pred': 1, 'score': 0.877342700958252, 'probs': [0.054989296942949295, 0.877342700958252, 0.06766801327466965]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0016.png'}, {'final_pred': 1, 'final_score': 0.9889967441558838, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.910489559173584, 'probs': [0.08296312391757965, 0.006547311320900917, 0.910489559173584]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9977695941925049, 'probs': [0.9977695941925049, 0.002230422105640173]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9889967441558838, 'probs': [0.0026011120062321424, 0.9889967441558838, 0.00811679009348154, 0.00028534038574434817]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0017.png'}, {'final_pred': 1, 'final_score': 0.9911621809005737, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9758456349372864, 'probs': [0.012706995010375977, 0.011447393335402012, 0.9758456349372864]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9991279244422913, 'probs': [0.9991279244422913, 0.0008720870828256011]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.9911621809005737, 'probs': [0.0004191198677290231, 0.9911621809005737, 0.008234787732362747, 0.00018386950250715017]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0018.png'}, {'final_pred': 0, 'final_score': 0.9981850981712341, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9229215979576111, 'probs': [0.06995793431997299, 0.007120476104319096, 0.9229215979576111]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9852843880653381, 'probs': [0.9852843880653381, 0.014715555123984814]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9981850981712341, 'probs': [0.9981850981712341, 0.0012281985254958272, 0.00015864385932218283, 0.00042808972648344934]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0019.png'}, {'final_pred': 0, 'final_score': 0.9522981643676758, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.9522981643676758, 'probs': [0.9522981643676758, 0.002543954411521554, 0.04515792056918144]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0020.png'}, {'final_pred': 0, 'final_score': 0.9954191446304321, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.6434285640716553, 'probs': [0.3547504246234894, 0.0018209666013717651, 0.6434285640716553]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9961674809455872, 'probs': [0.9961674809455872, 0.0038324412889778614]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9954191446304321, 'probs': [0.9954191446304321, 0.003806640859693289, 0.0002211379905929789, 0.0005530340713448822]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0021.png'}, {'final_pred': 0, 'final_score': 0.9908349514007568, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.6223533153533936, 'probs': [0.3739214241504669, 0.0037252188194543123, 0.6223533153533936]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9758185744285583, 'probs': [0.9758185744285583, 0.024181419983506203]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9908349514007568, 'probs': [0.9908349514007568, 0.00811957847326994, 0.0004540642839856446, 0.0005914602079428732]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0022.png'}, {'final_pred': 0, 'final_score': 0.9905316829681396, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8963890075683594, 'probs': [0.1009742021560669, 0.002636853139847517, 0.8963890075683594]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9619982242584229, 'probs': [0.9619982242584229, 0.03800176456570625]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9905316829681396, 'probs': [0.9905316829681396, 0.006741761229932308, 0.0009225566172972322, 0.0018039308488368988]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0023.png'}, {'final_pred': 0, 'final_score': 0.7035287618637085, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.7035287618637085, 'probs': [0.7035287618637085, 0.005394545383751392, 0.2910767197608948]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0024.png'}, {'final_pred': 0, 'final_score': 0.9970703125, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9382414221763611, 'probs': [0.059231288731098175, 0.002527275588363409, 0.9382414221763611]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9721794724464417, 'probs': [0.9721794724464417, 0.027820561081171036]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9970703125, 'probs': [0.9970703125, 0.002242363290861249, 0.00022241080296225846, 0.00046491148532368243]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0025.png'}, {'final_pred': 0, 'final_score': 0.9945883750915527, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9460312724113464, 'probs': [0.04618766903877258, 0.007781032007187605, 0.9460312724113464]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9925691485404968, 'probs': [0.9925691485404968, 0.007430859375745058]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9945883750915527, 'probs': [0.9945883750915527, 0.004377526231110096, 0.0003841168072540313, 0.0006499678129330277]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0026.png'}, {'final_pred': 0, 'final_score': 0.9738198518753052, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.937138020992279, 'probs': [0.05769425258040428, 0.005167766939848661, 0.937138020992279]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9982876181602478, 'probs': [0.9982876181602478, 0.0017123805591836572]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9738198518753052, 'probs': [0.9738198518753052, 0.024172868579626083, 0.0011098379036411643, 0.0008974599186331034]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0027.png'}, {'final_pred': 0, 'final_score': 0.9729928374290466, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9754466414451599, 'probs': [0.020480742678046227, 0.004072660580277443, 0.9754466414451599]}, {'model': 'Cluster', 'pred': 1, 'score': 0.659778892993927, 'probs': [0.3402211368083954, 0.659778892993927]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9729928374290466, 'probs': [0.9729928374290466, 0.01861688122153282, 0.00531792314723134, 0.001738700782880187, 0.00133369246032089]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0028.png'}, {'final_pred': 0, 'final_score': 0.991479754447937, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.940423846244812, 'probs': [0.05732068419456482, 0.0022554851602762938, 0.940423846244812]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9673227071762085, 'probs': [0.9673227071762085, 0.032677240669727325]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.991479754447937, 'probs': [0.991479754447937, 0.007293571252375841, 0.00039846618892624974, 0.0008282748167403042]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0029.png'}, {'final_pred': 0, 'final_score': 0.9063243269920349, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7836254239082336, 'probs': [0.21533843874931335, 0.001036152709275484, 0.7836254239082336]}, {'model': 'Cluster', 'pred': 1, 'score': 0.6834354400634766, 'probs': [0.31656455993652344, 0.6834354400634766]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9063243269920349, 'probs': [0.9063243269920349, 0.08002124726772308, 0.007698974572122097, 0.0029335811268538237, 0.0030218702740967274]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0030.png'}, {'final_pred': 0, 'final_score': 0.971327543258667, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.7341160178184509, 'probs': [0.2632089853286743, 0.002674996852874756, 0.7341160178184509]}, {'model': 'Cluster', 'pred': 0, 'score': 0.7584717273712158, 'probs': [0.7584717273712158, 0.2415282130241394]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.971327543258667, 'probs': [0.971327543258667, 0.024086041375994682, 0.0013917280593886971, 0.003194748191162944]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0031.png'}, {'final_pred': 1, 'final_score': 0.8609607815742493, 'path': [{'model': 'MonovsNonMono', 'pred': 1, 'score': 0.8609607815742493, 'probs': [0.04740600287914276, 0.8609607815742493, 0.09163319319486618]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0032.png'}, {'final_pred': 1, 'final_score': 0.7466588020324707, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.8323496580123901, 'probs': [0.11023839563131332, 0.05741186439990997, 0.8323496580123901]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9898185133934021, 'probs': [0.9898185133934021, 0.010181540623307228]}, {'model': 'Unclustered_RBCCount', 'pred': 1, 'score': 0.7466588020324707, 'probs': [0.24279357492923737, 0.7466588020324707, 0.008785750716924667, 0.0017618180718272924]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0033.png'}, {'final_pred': 0, 'final_score': 0.9882546663284302, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9426009058952332, 'probs': [0.050573114305734634, 0.006825902033597231, 0.9426009058952332]}, {'model': 'Cluster', 'pred': 0, 'score': 0.9591931104660034, 'probs': [0.9591931104660034, 0.04080687463283539]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9882546663284302, 'probs': [0.9882546663284302, 0.01003681868314743, 0.0006006755866110325, 0.0011077498784288764]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0034.png'}, {'final_pred': 0, 'final_score': 0.9894551634788513, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9714570045471191, 'probs': [0.02624126337468624, 0.002301808213815093, 0.9714570045471191]}, {'model': 'Cluster', 'pred': 1, 'score': 0.7115859389305115, 'probs': [0.28841400146484375, 0.7115859389305115]}, {'model': 'Cluster_RBCCount', 'pred': 0, 'score': 0.9894551634788513, 'probs': [0.9894551634788513, 0.0036160540767014027, 0.0030731079168617725, 0.0017938397359102964, 0.0020617737900465727]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0035.png'}, {'final_pred': 0, 'final_score': 0.9978064894676208, 'path': [{'model': 'MonovsNonMono', 'pred': 2, 'score': 0.9184052348136902, 'probs': [0.07894811779260635, 0.002646594075486064, 0.9184052348136902]}, {'model': 'Cluster', 'pred': 0, 'score': 0.6558628678321838, 'probs': [0.6558628678321838, 0.3441371023654938]}, {'model': 'Unclustered_RBCCount', 'pred': 0, 'score': 0.9978064894676208, 'probs': [0.9978064894676208, 0.0016506159445270896, 0.0001807436638046056, 0.00036204815842211246]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0036.png'}, {'final_pred': 0, 'final_score': 0.5049053430557251, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.5049053430557251, 'probs': [0.5049053430557251, 0.0023631486110389233, 0.492731511592865]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0037.png'}, {'final_pred': 0, 'final_score': 0.6205452680587769, 'path': [{'model': 'MonovsNonMono', 'pred': 0, 'score': 0.6205452680587769, 'probs': [0.6205452680587769, 0.0016200800891965628, 0.3778347074985504]}], 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x005_y012_cell_0038.png'}]

In [5]:
result[4]


{'final_pred': 3,
 'final_score': 0.7064077258110046,
 'path': [{'model': 'MonovsNonMono',
   'pred': 2,
   'score': 0.668097972869873,
   'probs': [0.3133552372455597, 0.018546834588050842, 0.668097972869873]},
  {'model': 'Cluster',
   'pred': 1,
   'score': 0.520740807056427,
   'probs': [0.479259192943573, 0.520740807056427]},
  {'model': 'Cluster_RBCCount',
   'pred': 3,
   'score': 0.7064077258110046,
   'probs': [0.0029736461583524942,
    0.004127341788262129,
    0.28442147374153137,
    0.7064077258110046,
    0.002069869777187705]}],
 'file': 'D:\\MMA_PipelineTest\\SingleCells\\tile_x003_y002_cell_0005.png'}

In [ ]:
import shutil
from pathlib import Path
import re

# Input and output base directories
src_root = Path(r"D:\MMA_LabelledData\Clustered_RBCCount")
dst_root = Path(r"D:\MMA_LabelledData\training_perslide\Clustered_RBCCount")

# Regex to extract slide info like "slide1_2"
slide_pattern = re.compile(r"slide(\d+)_(\d+)", re.IGNORECASE)

# Loop through top-level class folders (e.g., Unclustered_RBCCount)
for class_dir in src_root.iterdir():
    # if not class_dir.is_dir():
    #     print("cooked")
    #     continue

    # # Loop through subfolders (e.g., RBC_0, RBC_1, etc.)
    # for sub_dir in class_dir.iterdir():
    #     if not sub_dir.is_dir():
    #         print("super cooked")
    #         continue

    # Process all PNG files
    for img_path in class_dir.glob("*.png"):
        fname = img_path.name

        # Extract slide info
        m = slide_pattern.search(fname)
        if not m:
            print(f"Skipping (no slide info): {fname}")
            continue

        slideA, slideB = m.groups()
        slide_folder = f"Slide{slideA}-{slideB}"

        # Build destination directory
        dst_dir = dst_root / class_dir.name / slide_folder
        dst_dir.mkdir(parents=True, exist_ok=True)

        # Copy file
        shutil.copy2(img_path, dst_dir / fname)
        print(f"Copied → {dst_dir / fname}")


Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide1-2\tile_x001_y001_cell_0001_slide1_2.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide1-8\tile_x001_y001_cell_0001_slide1_8.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide2-3\tile_x001_y001_cell_0001_slide2_3.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide2-6\tile_x001_y001_cell_0001_slide2_6.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide1-2\tile_x001_y001_cell_0002_slide1_2.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide1-4\tile_x001_y001_cell_0002_slide1_4.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide1-8\tile_x001_y001_cell_0002_slide1_8.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide2-6\tile_x001_y001_cell_0002_slide2_6.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide2-7\tile_x001_y001_cell_0002_slide2_7.png
Copied → D:\MMA_LabelledData\training_perslide\RBC_0\Slide3-1\tile_x001_y001_cell_0002_slide3_1.png
